# tuf smoke test

In [ ]:
import datetime
from tuf.api.metadata import Metadata, Root, TOP_LEVEL_ROLE_NAMES

expires = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=365)
root = Root(version=1, expires=expires)
md = Metadata(signed=root)

assert isinstance(md.signed, Root) and md.signed.type == "root"
assert set(root.roles) == set(TOP_LEVEL_ROLE_NAMES)
assert not root.is_expired()
print("spec_version:", root.spec_version, "| roles:", sorted(root.roles))

In [ ]:
import datetime
from securesystemslib.signer import CryptoSigner
from tuf.api.metadata import Metadata, Root, TOP_LEVEL_ROLE_NAMES

expires = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=365)
root = Root(expires=expires)
md = Metadata(signed=root)

signer = CryptoSigner.generate_ed25519()  # in-memory test key, never persisted
for role in TOP_LEVEL_ROLE_NAMES:
    root.add_key(signer.public_key, role)

signature = md.sign(signer)
assert signature.keyid == signer.public_key.keyid
result = root.get_verification_result(Root.type, md.signed_bytes, md.signatures)
assert result.verified, result
md.verify_delegate("root", md)  # raises UnsignedMetadataError on failure
print("signed by", signature.keyid[:16], "... | verified:", result.verified)

In [ ]:
import datetime
from securesystemslib.signer import CryptoSigner
from tuf.api.metadata import Metadata, Root
from tuf.api.serialization.json import JSONSerializer

expires = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=365)
md = Metadata(signed=Root(expires=expires))
signer = CryptoSigner.generate_ed25519()
md.signed.add_key(signer.public_key, "root")
md.sign(signer)

pretty = md.to_bytes(JSONSerializer(compact=False))
md2 = Metadata.from_bytes(pretty)
assert isinstance(md2.signed, Root)
assert md2.to_dict() == md.to_dict()
assert md2.signed.get_verification_result("root", md2.signed_bytes, md2.signatures).verified
print("round-tripped", len(pretty), "bytes | version", md2.signed.version)